# 1.Abrir y examinar el archivo de datos.

In [29]:
#librerias necesarias pandas, ECM, Split del conjunto, modelos, metrica de exactitud

import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#Leemos el conjunto de los datos
df = pd.read_csv('users_behavior.csv')

print("\nInformación general:")
print(df.info())

print("\nEstadísticas descriptivas:")
print(df.describe())

#Examinamos los datos
print("Primeras 5 filas:")
print(df.head())

print("\nDistribución de la variable objetivo (is_ultra):")
print(df['is_ultra'].value_counts(normalize=True))

print("\nDistribución de los datos (filas, columnas): ")
print(df.shape)


Información general:
<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None

Estadísticas descriptivas:
             calls      minutes     messages       mb_used     is_ultra
count  3214.000000  3214.000000  3214.000000   3214.000000  3214.000000
mean     63.038892   438.208787    38.281269  17207.673836     0.306472
std      33.236368   234.569872    36.148326   7570.968246     0.461100
min       0.000000     0.000000     0.000000      0.000000     0.000000
25%      40.000000   274.575000     9.000000  12491.902500     0.000000
50%      62.000000   430.600000    30.000000  16943.235000     0.000000
75%      82.000000   

Vemos que el conjunto de los datos no tiene valores faltantes y los tipos de datos para cada columna son correctos.

# Segmenta los datos fuente en un conjunto de entrenamiento, uno de validación y uno de prueba.


Tomemos en cuenta que el conjunto de datos es pequeño, necesitamos que el tamaño de los subconjuntos de validación y prueba tengan un tamaño suficiente para garantizar la calidad del modelo, es por ello que elegimos el siguiente tamaño 60% entrenamiento, 20% validación y 20% prueba. Tenemos que tomar en cuenta que nuestra característica objetivo está desbalanceada por lo que es necesario usar stratify=target para que las proporciones de Smart y Ultra se mantengan similares en los 3 conjuntos

In [30]:
#definicion de variables features y target
features = df.drop(['is_ultra'], axis=1)  # Features
target = df['is_ultra']  # Target

#Primer Split para el subconjunto target de 60% y 40% en un temporal
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.4, random_state=12345, stratify=target)

#Segundo split para dividir el 40% de los datos en 20% y 20%
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.5, random_state=12345, stratify=target_temp)

print(f"Tamaño del conjunto de entrenamiento: {features_train.shape[0]} - 60%")
print(f"Tamaño del conjunto de validación: {features_valid.shape[0]} - 20%")
print(f"Tamaño del conjunto de prueba: {features_test.shape[0]} - 20%")

Tamaño del conjunto de entrenamiento: 1928 - 60%
Tamaño del conjunto de validación: 643 - 20%
Tamaño del conjunto de prueba: 643 - 20%


# Investigamos la calidad de diferentes modelos cambiando los hiperparámetros. 
Generemos una funcion para facilitar la evaluacion del modelo.

In [31]:
# Función para evaluar modelos
def evaluate_model(model, name):
    model.fit(features_train, target_train)
    predictions_valid = model.predict(features_valid)
    accuracy = accuracy_score(target_valid, predictions_valid)
    print(f"{name} - Exactitud en validación: {accuracy:.4f}")
    return accuracy

# 1. Decision Tree
best_tree_acc = 0
best_depth = 0
for depth in range(1, 11):
    model = DecisionTreeClassifier(max_depth=depth, random_state=12345)
    acc = evaluate_model(model, f"DecisionTree (depth={depth})")
    if acc > best_tree_acc:
        best_tree_acc = acc
        best_depth = depth

print(f"\nMejor Decision Tree: depth={best_depth} → {best_tree_acc:.4f}\n")

# 2. Random Forest
best_forest_acc = 0
best_est = 0
best_depth_forest = 0
for est in [10, 50, 100, 150]:
    for depth in [5, 10, 15, None]:
        model = RandomForestClassifier(n_estimators=est, max_depth=depth, random_state=12345)
        acc = evaluate_model(model, f"RandomForest (n_est={est}, depth={depth})")
        if acc > best_forest_acc:
            best_forest_acc = acc
            best_est = est
            best_depth_forest = depth

print(f"\nMejor Random Forest: n_estimators={best_est}, max_depth={best_depth_forest} → {best_forest_acc:.4f}")

# 3. Logistic Regression
model_lr = LogisticRegression(random_state=12345, solver='liblinear', max_iter=1000)
acc_lr = evaluate_model(model_lr, "Logistic Regression")

DecisionTree (depth=1) - Exactitud en validación: 0.7403
DecisionTree (depth=2) - Exactitud en validación: 0.7729
DecisionTree (depth=3) - Exactitud en validación: 0.7776
DecisionTree (depth=4) - Exactitud en validación: 0.7543
DecisionTree (depth=5) - Exactitud en validación: 0.7854
DecisionTree (depth=6) - Exactitud en validación: 0.7745
DecisionTree (depth=7) - Exactitud en validación: 0.7869
DecisionTree (depth=8) - Exactitud en validación: 0.8025
DecisionTree (depth=9) - Exactitud en validación: 0.7823
DecisionTree (depth=10) - Exactitud en validación: 0.7729

Mejor Decision Tree: depth=8 → 0.8025

RandomForest (n_est=10, depth=5) - Exactitud en validación: 0.7885
RandomForest (n_est=10, depth=10) - Exactitud en validación: 0.7994
RandomForest (n_est=10, depth=15) - Exactitud en validación: 0.7854
RandomForest (n_est=10, depth=None) - Exactitud en validación: 0.7869
RandomForest (n_est=50, depth=5) - Exactitud en validación: 0.8009
RandomForest (n_est=50, depth=10) - Exactitud en 

El mejor modelo es RandomForest con un numero de estimadores igual a 100 y profundidad de 10.

# Comprueba la calidad del modelo usando el conjunto de prueba.
Hagamos una prueba pra RandomForest


In [32]:
# Modelo final (ajusta con tus mejores hiperparámetros)
final_model = RandomForestClassifier(
    n_estimators=best_est, 
    max_depth=best_depth_forest, 
    random_state=12345
)

final_model.fit(features_train, target_train)
predictions_test = final_model.predict(features_test)
test_accuracy = accuracy_score(target_test, predictions_test)

print(f"\nExactitud del modelo final en el conjunto de PRUEBA: {test_accuracy:.4f}")


Exactitud del modelo final en el conjunto de PRUEBA: 0.8118


# Prueba de cordura al modelo
Veamos si nuestro modelo cumple con ser eficiente y mejor que dejar el pronostico al azar. Tenemos que considerar que el data set esta desbalanceado, si el modelo hace la prediccion seguramente va a predecir Smart porque el 70% de los usuarios usa este plan.

In [33]:
# Modelo dummy más frecuente (predice siempre la clase mayoritaria)
most_frequent = target_train.mode()[0]
predictions_dummy = [most_frequent] * len(target_test)
dummy_accuracy = accuracy_score(target_test, predictions_dummy)

print(f"Exactitud del modelo dummy (clase más frecuente): {dummy_accuracy:.4f}")
print(f"Exactitud de nuestro modelo: {test_accuracy:.4f}")

if test_accuracy > dummy_accuracy:
    print("✓ La prueba de cordura se superó. El modelo es mejor que predecir siempre la clase mayoritaria.")
else:
    print("✗ El modelo no pasó la prueba de cordura.")

Exactitud del modelo dummy (clase más frecuente): 0.6936
Exactitud de nuestro modelo: 0.8118
✓ La prueba de cordura se superó. El modelo es mejor que predecir siempre la clase mayoritaria.
